# 🎯 ML Leakage Detector - Kaggle Notebook Integration

**Detect data leakage in your Kaggle notebooks before submission!**

This notebook shows how to use the ML Leakage Detector to find and fix data leakage in your Kaggle competition solutions.

## 🚀 Quick Start

In [ ]:
# Install ML Leakage Detector
!pip install ml-leakage-detector -q

In [ ]:
import sys
sys.path.insert(0, '/kaggle/input/ml-leakage-detector/src')

## 📊 Scan Your Code for Data Leakage

In [ ]:
# Option 1: Scan a specific Python file
from src.detector import LeakageDetector

detector = LeakageDetector()
result = detector.detect_file('/kaggle/input/your-competition-code/preprocessing.py')

if result.has_leakage:
    print("⚠️ DATA LEAKAGE DETECTED!")
    for leak in result.leaks:
        print(f"  {leak.severity}: {leak.pattern} at line {leak.line_number}")
        print(f"    → {leak.suggestion}")
else:
    print("✅ No leakage detected - safe to submit!")

## 🔍 Quick Check: Paste Your Preprocessing Code

In [ ]:
# Paste your preprocessing code here and scan it
YOUR_CODE = """
# Example: Replace this with your preprocessing code
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('data.csv')
X = df.drop('target', axis=1)
y = df['target']

# ❌ LEAKAGE: Scaling before split!
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ❌ LEAKAGE: Sequential split!
X_train, X_test = X_scaled[:800], X_scaled[800:]
y_train, y_test = y[:800], y[800:]
"""

result = detector.detect(YOUR_CODE)

print("=" * 50)
print("SCAN RESULTS")
print("=" * 50)

if result.has_leakage:
    print(f"\n🚨 Found {len(result.leaks)} leakage issues:\n")
    
    # Sort by severity
    severity_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2}
    sorted_leaks = sorted(result.leaks, key=lambda x: severity_order.get(x.severity, 3))
    
    for leak in sorted_leaks:
        emoji = "🔥" if leak.severity == "CRITICAL" else "⚠️"
        print(f"{emoji} {leak.severity}: {leak.pattern}")
        print(f"   Line {leak.line_number}: {leak.code_snippet[:50]}...")
        print(f"   💡 Fix: {leak.suggestion}\n")
else:
    print("✅ No leakage detected!")

## ✅ Corrected Code Example

In [ ]:
# Here's the CORRECT way to do preprocessing
CORRECT_CODE = """
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('data.csv')
X = df.drop('target', axis=1)
y = df['target']

# ✅ CORRECT: Split BEFORE scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ✅ CORRECT: Fit scaler ONLY on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use transform, not fit_transform!
"""

result = detector.detect(CORRECT_CODE)

print("Testing CORRECT code:")
print("=" * 50)
if result.has_leakage:
    print(f"⚠️ Found {len(result.leaks)} issues")
else:
    print("✅ No leakage - This is the correct approach!")

## 📋 Pre-Submission Checklist

In [ ]:
def pre_submission_check(code):
    """Run all checks before submitting your notebook"""
    result = detector.detect(code)
    
    checks = {
        "Has Leakage": result.has_leakage,
        "Critical Issues": len([l for l in result.leaks if l.severity == "CRITICAL"]),
        "Medium Issues": len([l for l in result.leaks if l.severity == "MEDIUM"]),
    }
    
    return checks

# Run checklist on your final code
checklist = pre_submission_check(CORRECT_CODE)

print("PRE-SUBMISSION CHECKLIST")
print("=" * 40)
for check, status in checklist.items():
    if isinstance(status, bool):
        emoji = "✅" if not status else "❌"
    else:
        emoji = "✅" if status == 0 else "⚠️"
    print(f"{emoji} {check}: {status}")

## 🎯 What We Detect

In [ ]:
print("""
DETECTION PATTERNS:
====================

🔥 CRITICAL (Fix before submission!)
  • imputation_before_split   - Using full data mean/median
  • scaling_before_split     - Fitting scaler on all data
  • target_encoding_before_split - Using target for features
  • feature_selection_before_split - Selecting features with target

⚠️ MEDIUM (Review carefully)
  • sequential_split          - Manual slicing instead of random split
  • train_test_contamination - Data leakage between splits
""")

---

**💡 Pro Tip:** Run this detector on your notebook before every submission!

*Built for Kaggle users by Dennis Omboga* 🏆